# T04: Evaluation

Ein RAG-System, das bei ein paar ausgewählten Fragen zu funktionieren *scheint*, kann trotzdem oft scheitern. Evaluation macht aus "scheint besser" eine Zahl, die sich über Änderungen hinweg vergleichen lässt:

```
Change a parameter  ──►  Measure  ──►  Decide based on data
```

Code, Prompts und Dokumente sind auf Englisch (Begründung in T01).

## Warum systematisch evaluieren?

In T01 beschrieb die Baseline manchmal die nicht existierende Lara-Palette, zitierte den nicht verifizierten GWP-Wert von 2021 statt der EPD oder übernahm die 68 % CO₂-Reduktion von tesa ohne Hinweis. Hat ein neuer Prompt oder eine andere Chunk-Grösse das behoben, und ist dabei etwas anderes kaputtgegangen? Das zeigen nur ein festes Testset und Metriken.

Jede Stufe der Pipeline scheitert anders, deshalb zielen die Metriken auf einzelne Stufen:

| Stufe | Wenn sie scheitert |
|---|---|
| Chunking | Fakten auf mehrere Chunks verteilt, Tabellen zerrissen |
| Embedding und Retrieval | Relevante Chunks nicht gefunden oder weit unten |
| Generierung | Halluzination, Kontext ignoriert, unvollständige Antwort |

## RAGAS

[RAGAS](https://docs.ragas.io) evaluiert RAG-Pipelines mit einem LLM als Bewerter. Statt einer Note von 0 bis 5 zerlegt es die Antwort in einzelne Aussagen und prüft jede gegen den gefundenen Kontext:

```
Answer: "The Logypal 1 EPD is registered as S-P-09135 and verified by Bureau Veritas."

  Claim 1: "registered as S-P-09135"   -> supported by context?  ✓
  Claim 2: "verified by Bureau Veritas" -> supported by context?  ✓

  Faithfulness = 2 / 2 = 1.0
```

So fallen auch teilweise Halluzinationen auf, etwa eine richtige Zahl mit einer erfundenen Prüfstelle.

| Metrik | Braucht Referenzantwort? | Erkennt |
|---|---|---|
| `Faithfulness` | Nein | Aussagen, die der gefundene Kontext nicht stützt |
| `AnswerRelevancy` | Nein | Antworten am Thema vorbei oder ausweichend |
| `ContextPrecision` | Ja | Irrelevante Chunks vor relevanten |
| `ContextRecall` | Ja | Fakten, die im gefundenen Kontext fehlen |

Die Referenzantworten stammen aus dem von Hand geschriebenen Testset `EVALUATION_QUERIES` in `ki_kompass_rag/evaluation.py`. Der Bewerter macht pro Beispiel und Metrik mehrere LLM-Aufrufe und hat eigene Verzerrungen: Die Scores sind Näherungswerte, keine absolute Wahrheit.

## Setup

Benötigt `OPENAI_API_KEY` (lokal: in `.env.local` im Hauptverzeichnis des Repos). Zuerst T01 ausführen: Dort entsteht der Vector Store, den die späteren Notebooks wiederverwenden. RAGAS verwendet `gpt-4o-mini` als Bewerter.

In [ ]:
import math
import sys

from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings as LangChainOpenAIEmbeddings
from loguru import logger
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import AnswerRelevancy, ContextPrecision, ContextRecall, Faithfulness

from conversational_toolkit.evaluation import Evaluator
from conversational_toolkit.evaluation.adapters import evaluate_with_ragas
from ki_kompass_rag.baseline_rag import (
    RETRIEVER_TOP_K,
    SYSTEM_PROMPT,
    build_agent,
    build_llm,
    load_or_build_vector_store,
)
from ki_kompass_rag.evaluation import EVALUATION_QUERIES

logger.remove()
logger.add(sys.stderr, level="WARNING")

vector_store, embedding_model = await load_or_build_vector_store()
agent = build_agent(
    vector_store,
    embedding_model,
    build_llm("openai"),
    top_k=RETRIEVER_TOP_K,
    system_prompt=SYSTEM_PROMPT,
)

# RAGAS's default limit of 3072 output tokens cuts off the judge's JSON on long answers.
ragas_llm = LangchainLLMWrapper(
    ChatOpenAI(model="gpt-4o-mini", max_completion_tokens=8192)
)
ragas_embeddings = LangChainOpenAIEmbeddings(model="text-embedding-3-small")


def print_scores(report, queries: list[str]) -> None:
    for name, score in report.summary().items():
        print(f"{name:<20}{score:.3f}")
    print("\n" + "".join(f"{r.metric_name:>20}" for r in report.results) + "  query")
    for i, query in enumerate(queries):
        scores = [r.per_sample_scores[i] for r in report.results]
        cells = "".join(
            f"{'n/a' if math.isnan(s) else format(s, '.2f'):>20}" for s in scores
        )
        print(f"{cells}  {query[:60]}")

## Teil 1: Qualität der Generierung (ohne Referenzantwort)

`Evaluator.build_samples_from_agent()` führt den Agenten für jede Testfrage aus und speichert Antwort, gefundene Chunks und Referenzantwort. Faithfulness und AnswerRelevancy verwenden nur Frage, Antwort und Chunks. Das dauert einige Minuten.

In [ ]:
queries = [q["query"] for q in EVALUATION_QUERIES]
samples = await Evaluator.build_samples_from_agent(
    agent=agent,
    queries=queries,
    ground_truth_answers=[q["ground_truth_answer"] for q in EVALUATION_QUERIES],
)

report_generation = evaluate_with_ragas(
    samples=samples,
    metrics=[Faithfulness(), AnswerRelevancy(strictness=1)],
    llm=ragas_llm,
    embeddings=ragas_embeddings,
)
print_scores(report_generation, queries)

Tiefe Faithfulness bei einer Frage heisst: Die Antwort enthält Aussagen, die die gefundenen Chunks nicht stützen. Diese Antwort lesen: `samples[i].answer`.

## Teil 2: Qualität des Retrievals (Referenzantwort nötig)

Metriken zur Generierung sagen nichts darüber, ob der Retriever die *richtigen* Chunks gefunden hat. ContextPrecision und ContextRecall vergleichen die gefundenen Chunks mit den Referenzantworten in `samples`.

| Metrik | 1.0 heisst | Tief heisst |
|---|---|---|
| ContextPrecision | Alle gefundenen Chunks sind relevant, die besten zuerst | Störende oder schlecht sortierte Chunks |
| ContextRecall | Jeder Fakt der Referenzantwort steht im Kontext | Der Retriever verpasst wichtige Chunks |

In [ ]:
report_retrieval = evaluate_with_ragas(
    samples=samples,
    metrics=[ContextPrecision(), ContextRecall()],
    llm=ragas_llm,
    embeddings=ragas_embeddings,
)
print_scores(report_retrieval, queries)

## Zusammenfassung

| Metrik | Referenzantwort? | Sagt aus |
|---|---|---|
| `Faithfulness` | Nein | Stützen sich die Antworten auf den gefundenen Kontext? |
| `AnswerRelevancy` | Nein | Gehen die Antworten auf die Frage ein? |
| `ContextPrecision` | Ja | Sind die gefundenen Chunks relevant und gut sortiert? |
| `ContextRecall` | Ja | Enthält der Kontext alle nötigen Fakten? |

Tiefe Scores bei der Generierung: Prompt oder LLM anschauen. Tiefe Scores beim Retrieval: Chunking, Embeddings oder `top_k` anschauen (T02, T07). T05 baut ein grösseres Testset.